# E0+ — Ablation tăng dần từ E0 (R0 → R4)
Nhóm 9 — VSL 34 lớp, LOSO 4 fold. Mỗi lần chạy chỉ thêm **một** thay đổi so với lần trước, để biết thay đổi nào có ích.

| Run | Thay đổi | 
|---|---|
| R0 | Đúng cấu hình E0 (kiểm tra tái lập ~40,8%) |
| R1 | R0 + 300 epoch (E0 chưa hội tụ ở epoch 100) |
| R2 | R1 + z-score từng chiều (fit trên tập fit, lưu trong model) |
| R3 | R2 + ensemble 5 seed |
| R4 | R3 + augmentation nhẹ: tịnh tiến x,y ±0,03, co giãn ±5% quanh tâm tay. **Không xoay, không frame jitter** |

Giữ nguyên ở mọi run: MLP 128→64 (ReLU, Dropout 0.3), Adam 1e-3, batch 32, ReduceLROnPlateau, validation 15% phân tầng, chọn checkpoint theo val acc, flip pair (1−x) khi train, TTA trung bình softmax, khung index 22, **không chuẩn hóa hình học**.

Latency đo trên cùng thiết bị với model (GPU nếu có, giống E1–E3), có `torch.cuda.synchronize()`. Độ lệch chuẩn giữa các fold chia n−1 (`ddof=1`).

Thời gian chạy ước tính khoảng 35–40 lần một lượt E0. Có thể chạy một phần bằng cách sửa `RUNS_TO_RUN`.

In [1]:
import re, json, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên: {DEVICE}")
_c = list(Path("/kaggle/input").rglob("landmarks/landmarks/raw"))
if not _c:
    raise FileNotFoundError("Không tìm thấy landmarks/landmarks/raw trong /kaggle/input")
LANDMARK_DIR = _c[0]
PEOPLE = ("hau", "khoi", "tai", "vy")
FNAME_RE = re.compile(r"^([a-z_]+)_([a-z]+)_([AB])_(\d+)\.npy$")
OUT_DIR = Path("/kaggle/working"); CKPT_DIR = OUT_DIR / "checkpoints_E0plus"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = OUT_DIR / "E0plus_ablation.json"
PLOT_DIR = OUT_DIR / "plots_E0plus"; PLOT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_CKPT_RUNS = ("R2",)   # chỉ lưu checkpoint của run được chọn (R2); thêm run khác nếu cần
MAU_XANH, MAU_CAM = "#2a78d6", "#eb6834"

FRAME_INDEX, LEARNING_RATE, BATCH_SIZE, VAL_RATIO, SEED, VALID_EPS = 22, 1e-3, 32, 0.15, 42, 1e-7
LATENCY_WARMUP, LATENCY_RUNS = 20, 100

RUNS = {
    "R0": dict(epochs=100, zscore=False, ensemble=1, aug=False),
    "R1": dict(epochs=300, zscore=False, ensemble=1, aug=False),
    "R2": dict(epochs=300, zscore=True,  ensemble=1, aug=False),
    "R3": dict(epochs=300, zscore=True,  ensemble=5, aug=False),
    "R4": dict(epochs=300, zscore=True,  ensemble=5, aug=True),
}
RUNS_TO_RUN = ["R0", "R1", "R2", "R3", "R4"]
AUG_SHIFT, AUG_SCALE = 0.03, 0.05

Đang chạy trên: cuda


In [2]:
records = []
for f in sorted(LANDMARK_DIR.glob("*/*.npy")):
    m = FNAME_RE.match(f.name)
    if m:
        code, person, block, seq = m.groups()
        records.append({"code": code, "person": person, "arr": np.load(f)})
ALL_CLASSES = sorted(set(r["code"] for r in records))
CODE_TO_IDX = {c: i for i, c in enumerate(ALL_CLASSES)}
print(f"Tổng số mẫu: {len(records)} | Số lớp: {len(ALL_CLASSES)}")

Tổng số mẫu: 640 | Số lớp: 34


In [3]:
class MLPClassifier(nn.Module):
    """Giống hệt E0; thêm tùy chọn z-score (buffer) ở đầu vào."""
    def __init__(self, in_dim, num_classes, mean=None, std=None):
        super().__init__()
        self.use_z = mean is not None
        if self.use_z:
            self.register_buffer("mean", torch.as_tensor(mean, dtype=torch.float32))
            self.register_buffer("std", torch.as_tensor(std, dtype=torch.float32))
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        if self.use_z:
            x = (x - self.mean) / self.std
        return self.net(x)


def flip_horizontal(X):
    """x' = 1 - x trên tọa độ ảnh thô; tay mất landmark giữ 0; 2 tay thì đổi slot (giống E0)."""
    h = X.reshape(X.shape[0], -1, 21, 3).copy()
    v = np.abs(h).sum(axis=(-1, -2)) > VALID_EPS
    h[..., 0] = np.where(v[..., None], 1.0 - h[..., 0], h[..., 0])
    if h.shape[1] == 2:
        h = h[:, ::-1].copy()
    return h.reshape(X.shape).astype(np.float32)


def augment_batch(xb, gen):
    """Augmentation nhẹ trên GPU/CPU: tịnh tiến x,y và co giãn quanh tâm (x,y) của tay.
    Không xoay (tọa độ x,y khác tỉ lệ nên xoay sẽ làm méo bàn tay). Tay mất landmark giữ 0."""
    B = xb.shape[0]
    h = xb.view(B, -1, 21, 3).clone()
    valid = (h.abs().sum(dim=(-1, -2)) > VALID_EPS)                      # (B, H)
    center = h[..., :2].mean(dim=2, keepdim=True)
    scale = 1 + (torch.rand(B, 1, 1, 1, generator=gen) * 2 - 1).to(xb.device) * AUG_SCALE
    shift = (torch.rand(B, 1, 1, 2, generator=gen) * 2 - 1).to(xb.device) * AUG_SHIFT
    h[..., :2] = (h[..., :2] - center) * scale + center + shift
    h[..., 2:] = h[..., 2:] * scale
    h = h * valid[..., None, None]
    return h.view(B, -1)


def split_train_validation(y, seed):
    rng = np.random.default_rng(seed)
    tr, va = [], []
    for c in np.unique(y):
        idx = np.flatnonzero(y == c); rng.shuffle(idx)
        n = min(max(1, int(len(idx) * VAL_RATIO)), len(idx) - 1)
        va.extend(idx[:n]); tr.extend(idx[n:])
    return np.asarray(tr), np.asarray(va)


def train_one(X_train, y_train, cfg, fold_seed, model_seed):
    """Huấn luyện 1 model theo đúng quy trình E0. model_seed == fold_seed ở member đầu tiên
    để R0 tái lập E0."""
    np.random.seed(model_seed); torch.manual_seed(model_seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(model_seed)
    tr, va = split_train_validation(y_train, fold_seed)          # split cố định theo fold
    X_fit, y_fit = X_train[tr], y_train[tr]
    X_val, y_val = X_train[va], y_train[va]
    X_fit = np.concatenate((X_fit, flip_horizontal(X_fit))); y_fit = np.concatenate((y_fit, y_fit))

    mean = std = None
    if cfg["zscore"]:
        mean, std = X_fit.mean(0), X_fit.std(0) + 1e-6
    model = MLPClassifier(X_fit.shape[1], len(ALL_CLASSES), mean, std).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=15, min_lr=1e-5)
    crit = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(model_seed)
    aug_gen = torch.Generator().manual_seed(model_seed + 999)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_fit).float(), torch.from_numpy(y_fit).long()),
                        batch_size=BATCH_SIZE, shuffle=True, generator=gen)
    Xv = torch.from_numpy(X_val).float().to(DEVICE); yv = torch.from_numpy(y_val).long().to(DEVICE)

    best_acc, best_ep, best_state = -1.0, 0, None
    history = {"train_loss": [], "val_acc": []}
    for ep in range(cfg["epochs"]):
        model.train(); total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if cfg["aug"]:
                xb = augment_batch(xb, aug_gen)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb); loss.backward(); opt.step()
            total += loss.item() * xb.size(0)
        model.eval()
        with torch.inference_mode():
            acc = float((model(Xv).argmax(1) == yv).float().mean())
        history["train_loss"].append(total / len(X_fit)); history["val_acc"].append(acc)
        sched.step(acc)
        if acc > best_acc:
            best_acc, best_ep = acc, ep + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state); model.eval()
    return model, best_ep, best_acc, history


def tta_probs(models, X, device):
    Xo = torch.from_numpy(X).float().to(device); Xf = torch.from_numpy(flip_horizontal(X)).float().to(device)
    p = 0
    for m in models:
        p = p + torch.softmax(m(Xo), 1) + torch.softmax(m(Xf), 1)
    return p / (2 * len(models))


def measure_latency(models, x):
    """Đo trên DEVICE (GPU nếu có), 1 mẫu, toàn bộ TTA của cả ensemble.
    Tensor gốc/lật tạo sẵn trên thiết bị trước vòng đo (giống E1)."""
    for m in models: m.eval()
    Xo = torch.from_numpy(x).float().to(DEVICE); Xf = torch.from_numpy(flip_horizontal(x)).float().to(DEVICE)

    def step():
        p = 0
        for m in models:
            p = p + torch.softmax(m(Xo), 1) + torch.softmax(m(Xf), 1)
        return p / (2 * len(models))

    def sync():
        if DEVICE.type == "cuda": torch.cuda.synchronize()

    with torch.inference_mode():
        for _ in range(LATENCY_WARMUP): step()
        sync(); t = []
        for _ in range(LATENCY_RUNS):
            sync(); s = time.perf_counter(); step(); sync(); t.append((time.perf_counter() - s) * 1000)
    return float(np.mean(t))

In [4]:
def plot_run_curves(name, fold_histories):
    """2x2: train loss & val accuracy theo epoch cho 4 fold (model đầu tiên của ensemble)."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (person, h, best_ep) in zip(axes.ravel(), fold_histories):
        ep = range(1, len(h["train_loss"]) + 1)
        ax.plot(ep, h["train_loss"], color=MAU_XANH, lw=2, label="Train loss")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Train loss", color=MAU_XANH)
        ax2 = ax.twinx()
        ax2.plot(ep, h["val_acc"], color=MAU_CAM, lw=2, label="Val accuracy")
        ax2.axvline(best_ep, color="gray", ls="--", lw=1)
        ax2.set_ylim(0, 1); ax2.set_ylabel("Val accuracy", color=MAU_CAM)
        ax.set_title(f"Fold test={person} (checkpoint epoch {best_ep})")
        l1, t1 = ax.get_legend_handles_labels(); l2, t2 = ax2.get_legend_handles_labels()
        ax.legend(l1 + l2, t1 + t2, loc="center right", fontsize=8)
    fig.suptitle(f"E0+ {name} — Train loss & Val accuracy")
    fig.tight_layout(); fig.savefig(PLOT_DIR / f"E0plus_{name}_curves.png", dpi=150); plt.close(fig)


def plot_confusion(name, yt, yp):
    cm = confusion_matrix(yt, yp, labels=range(len(ALL_CLASSES))).astype(float)
    cm = 100 * cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, vmin=0, vmax=100, cmap="viridis")
    ax.set_xticks(range(len(ALL_CLASSES))); ax.set_xticklabels(ALL_CLASSES, rotation=90, fontsize=7)
    ax.set_yticks(range(len(ALL_CLASSES))); ax.set_yticklabels(ALL_CLASSES, fontsize=7)
    ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
    fig.colorbar(im, ax=ax, label="Row-normalized (%)")
    ax.set_title(f"E0+ {name} — Pooled OOF confusion matrix (%)")
    fig.tight_layout(); fig.savefig(PLOT_DIR / f"E0plus_{name}_confusion.png", dpi=150); plt.close(fig)


def run(name, cfg):
    print(f"\n===== {name}: {cfg} =====")
    folds, yt_all, yp_all, fold_histories = [], [], [], []
    for k, test_person in enumerate(PEOPLE):
        fold_seed = SEED + k
        trr = [r for r in records if r["person"] != test_person]; ter = [r for r in records if r["person"] == test_person]
        X_train = np.stack([r["arr"] for r in trr])[:, FRAME_INDEX, :].astype(np.float32)
        y_train = np.array([CODE_TO_IDX[r["code"]] for r in trr])
        X_test = np.stack([r["arr"] for r in ter])[:, FRAME_INDEX, :].astype(np.float32)
        y_test = np.array([CODE_TO_IDX[r["code"]] for r in ter])

        members = [train_one(X_train, y_train, cfg, fold_seed, fold_seed + 1000 * s) for s in range(cfg["ensemble"])]
        models = [m for m, _, _, _ in members]
        fold_histories.append((test_person, members[0][3], members[0][1]))
        with torch.inference_mode():
            y_pred = tta_probs(models, X_test, DEVICE).argmax(1).cpu().numpy()
        acc = float((y_pred == y_test).mean()); f1 = float(f1_score(y_test, y_pred, average="macro"))
        lat = measure_latency(models, X_test[:1])
        yt_all.append(y_test); yp_all.append(y_pred)
        folds.append({"test_person": test_person, "accuracy": acc, "macro_f1": f1, "latency_ms": lat,
                      "best_epochs": [e for _, e, _, _ in members]})
        if name in SAVE_CKPT_RUNS:
            torch.save({"run": name, "config": cfg, "test_person": test_person, "classes": ALL_CLASSES,
                        "frame_index": FRAME_INDEX, "state_dicts": [m.state_dict() for m in models]},
                       CKPT_DIR / f"{name}_fold{k}_{test_person}.pt")
        print(f"  test={test_person}: acc={acc:.3f} | F1={f1:.3f} | lat={lat:.3f} ms | best epoch={folds[-1]['best_epochs']}")
    accs = [f["accuracy"] for f in folds]; f1s = [f["macro_f1"] for f in folds]; lats = [f["latency_ms"] for f in folds]
    yt, yp = np.concatenate(yt_all), np.concatenate(yp_all)
    summ = {"run": name, "config": cfg, "folds": folds,
            "accuracy_mean": float(np.mean(accs)), "accuracy_std": float(np.std(accs, ddof=1)),
            "macro_f1_mean": float(np.mean(f1s)), "macro_f1_std": float(np.std(f1s, ddof=1)),
            "latency_mean_ms": float(np.mean(lats)), "latency_std_ms": float(np.std(lats, ddof=1)),
            "latency_device": DEVICE.type, "std": "sample std across folds (ddof=1)",
            "pooled_oof_accuracy": float((yt == yp).mean()),
            "pooled_oof_macro_f1": float(f1_score(yt, yp, average="macro"))}
    plot_run_curves(name, fold_histories)
    if name in SAVE_CKPT_RUNS:
        plot_confusion(name, yt, yp)
    print(f"  => {name}: acc={summ['accuracy_mean']:.3f} ± {summ['accuracy_std']:.3f} | "
          f"F1={summ['macro_f1_mean']:.3f} | lat={summ['latency_mean_ms']:.3f} ms")
    return summ


all_results = {}
for name in RUNS_TO_RUN:
    all_results[name] = run(name, RUNS[name])
    with open(RESULT_PATH, "w", encoding="utf-8") as f:    # lưu sau mỗi run, lỡ bị ngắt vẫn còn kết quả
        json.dump(all_results, f, ensure_ascii=False, indent=2)


===== R0: {'epochs': 100, 'zscore': False, 'ensemble': 1, 'aug': False} =====
  test=hau: acc=0.406 | F1=0.392 | lat=0.524 ms | best epoch=[87]
  test=khoi: acc=0.500 | F1=0.478 | lat=0.551 ms | best epoch=[87]
  test=tai: acc=0.356 | F1=0.328 | lat=0.857 ms | best epoch=[91]
  test=vy: acc=0.369 | F1=0.337 | lat=0.547 ms | best epoch=[86]
  => R0: acc=0.408 ± 0.065 | F1=0.384 | lat=0.620 ms

===== R1: {'epochs': 300, 'zscore': False, 'ensemble': 1, 'aug': False} =====
  test=hau: acc=0.406 | F1=0.392 | lat=0.564 ms | best epoch=[87]
  test=khoi: acc=0.525 | F1=0.519 | lat=0.540 ms | best epoch=[139]
  test=tai: acc=0.369 | F1=0.333 | lat=0.519 ms | best epoch=[126]
  test=vy: acc=0.369 | F1=0.337 | lat=0.543 ms | best epoch=[86]
  => R1: acc=0.417 ± 0.074 | F1=0.395 | lat=0.541 ms

===== R2: {'epochs': 300, 'zscore': True, 'ensemble': 1, 'aug': False} =====
  test=hau: acc=0.463 | F1=0.451 | lat=0.611 ms | best epoch=[27]
  test=khoi: acc=0.562 | F1=0.543 | lat=0.621 ms | best epoch=

In [5]:
print(f"{'Run':<4} {'Accuracy':>16} {'Macro-F1':>16} {'Latency (ms)':>13}   " + "  ".join(PEOPLE))
for n, r in all_results.items():
    per = "  ".join(f"{f['accuracy']:.3f}" for f in r["folds"])
    print(f"{n:<4} {r['accuracy_mean']:.3f} ± {r['accuracy_std']:.3f}   {r['macro_f1_mean']:.3f} ± {r['macro_f1_std']:.3f}"
          f"   {r['latency_mean_ms']:>10.3f}   {per}")
names = list(all_results)
x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(9, 5))
for off, key, color, lab in ((-w / 2, "accuracy", MAU_XANH, "Accuracy"), (w / 2, "macro_f1", MAU_CAM, "Macro-F1")):
    means = [100 * all_results[n][f"{key}_mean"] for n in names]
    stds = [100 * all_results[n][f"{key}_std"] for n in names]
    bars = ax.bar(x + off, means, w, yerr=stds, capsize=4, color=color, label=lab)
    ax.bar_label(bars, fmt="%.1f", fontsize=8, padding=2)
ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylim(0, 100); ax.set_ylabel("%")
ax.set_title("E0+ ablation — LOSO 4 fold (mean ± std, ddof=1)"); ax.legend()
fig.tight_layout(); fig.savefig(PLOT_DIR / "E0plus_ablation_summary.png", dpi=150); plt.close(fig)
print(f"Đã lưu biểu đồ vào {PLOT_DIR}")

best = max(all_results, key=lambda n: all_results[n]["accuracy_mean"])
print(f"\nRun tốt nhất theo accuracy trung bình LOSO: {best}")
print("Lưu ý: chọn run tốt nhất sau khi xem kết quả test là lựa chọn post-hoc — báo cáo cả bảng, không chỉ run tốt nhất.")

Run          Accuracy         Macro-F1  Latency (ms)   hau  khoi  tai  vy
R0   0.408 ± 0.065   0.384 ± 0.069        0.620   0.406  0.500  0.356  0.369
R1   0.417 ± 0.074   0.395 ± 0.087        0.541   0.406  0.525  0.369  0.369
R2   0.514 ± 0.067   0.501 ± 0.076        0.601   0.463  0.562  0.450  0.581
R3   0.494 ± 0.064   0.480 ± 0.077        2.578   0.444  0.525  0.438  0.569
R4   0.509 ± 0.060   0.501 ± 0.073        2.649   0.463  0.537  0.456  0.581
Đã lưu biểu đồ vào /kaggle/working/plots_E0plus

Run tốt nhất theo accuracy trung bình LOSO: R2
Lưu ý: chọn run tốt nhất sau khi xem kết quả test là lựa chọn post-hoc — báo cáo cả bảng, không chỉ run tốt nhất.
